In [ ]:
# Cell 1: Cài đặt thư viện
!pip install web3 py-solc-x gradio -q
print("✅ Đã cài đặt xong thư viện.")

✅ Đã cài đặt xong thư viện.


In [ ]:
import json, os
from web3 import Web3
from google.colab import files
import gradio as gr

INFURA_URL = "https://sepolia.infura.io/v3/7e77823ee10a44ca81d65804ba33b9c8"
SHOP_ADDRESS = "0x47D821A5B2a86bBECeb07bd05E1Ed32C196E732e"
ARBITER_KEY = "68b75cf1c0de408e52091266d0ba8afe0e19d26dc182c3d7c6a1ed5b462a8d5b"

w3 = Web3(Web3.HTTPProvider(INFURA_URL))
arbiter = w3.eth.account.from_key(ARBITER_KEY)

if not os.path.exists("ShopABI.json"): files.upload()
with open("ShopABI.json") as f: abi = json.load(f)
contract = w3.eth.contract(address=w3.to_checksum_address(SHOP_ADDRESS), abi=abi)
print(f"✅ Arbiter sẵn sàng: {arbiter.address}")

✅ Arbiter sẵn sàng: 0xc9f30A0287ad97691335281Cc27189a95f4b5927


In [ ]:
# --- LOGIC XỬ LÝ ---

# 1. Lấy danh sách đơn ĐANG tranh chấp (Cần xử lý)
def get_active_disputes():
    cases = []
    try:
        count = contract.functions.orderCounter().call()
        for i in range(count):
            o = contract.functions.orders(i).call()
            # State = 3 là DISPUTED
            if o[5] == 3:
                p = contract.functions.products(o[1]).call()
                # Chỉ hiện đơn thuộc thẩm quyền của mình
                if p[6] == arbiter.address:
                    amt = float(w3.from_wei(o[4], 'ether'))
                    cases.append(
                        f"🔥 VỤ KIỆN #{o[0]} | SP: {p[1]}\n"
                        f"   💰 Giá trị: {amt} ETH\n"
                        f"   📝 Lý do: '{o[6]}'\n"
                        f"-----------------------------------"
                    )
        return "\n".join(cases) if cases else "✅ Hiện không có vụ kiện nào cần xử lý."
    except Exception as e: return f"❌ Lỗi: {e}"

# 2. (MỚI) Xem Lịch sử Phán quyết (Đã xử xong)
def view_judgment_history():
    history = []
    try:
        count = contract.functions.orderCounter().call()

        # Header
        header = f"{'MÃ VỤ':<8} | {'KẾT QUẢ PHÁN QUYẾT':<30} | {'LÝ DO TRANH CHẤP'}"
        history.append(header)
        history.append("-" * 90)

        for i in range(count):
            o = contract.functions.orders(i).call()
            # o[6] là lý do dispute. Nếu khác rỗng ("") nghĩa là đơn này từng có tranh chấp
            if o[6]:
                p = contract.functions.products(o[1]).call()
                if p[6] == arbiter.address:

                    verdict = ""
                    # State 4 = REFUNDED (Buyer Thắng)
                    if o[5] == 4:
                        verdict = "🏆 BUYER THẮNG (Đã hoàn tiền)"
                    # State 2 = COMPLETED (Seller Thắng - vì Dispute xong mà về Completed tức là tiền về Seller)
                    elif o[5] == 2:
                        verdict = "🏆 SELLER THẮNG (Đã trả tiền)"
                    else:
                        continue # Bỏ qua đơn đang xử lý (State 3) hoặc chưa xong

                    line = f"#{o[0]:<7} | {verdict:<30} | '{o[6]}'"
                    history.append(line)

        if len(history) <= 2: return "📭 Chưa có lịch sử phán quyết nào."
        return "\n".join(history)
    except Exception as e: return f"❌ Lỗi tải lịch sử: {e}"

# 3. Công cụ Thẩm định (Gợi ý)
def analyze_evidence(days_passed, seller_received_goods):
    try:
        days = int(days_passed)
        if seller_received_goods:
            if days <= 7:
                return "💡 GỢI Ý: Buyer trả hàng ĐÚNG hạn.\n👉 PHÁN QUYẾT: HOÀN TIỀN CHO BUYER."
            else:
                return "💡 GỢI Ý: Buyer trả hàng QUÁ hạn, nhưng Seller đã nhận.\n👉 PHÁN QUYẾT: HOÀN TIỀN CHO BUYER (Đồng thuận)."
        else:
            if days > 7:
                return "💡 GỢI Ý: Quá 7 ngày Seller chưa nhận được hàng.\n👉 PHÁN QUYẾT: SELLER THẮNG."
            else:
                return f"⏳ GỢI Ý: Mới {days} ngày. Chưa đủ căn cứ xử thua.\n👉 HÀNH ĐỘNG: Yêu cầu chờ thêm."
    except: return "❌ Nhập số ngày hợp lệ."

# 4. Ra Phán Quyết (Blockchain)
def execute_judgment(order_id, verdict):
    try:
        if not order_id: return "⚠️ Thiếu Mã vụ kiện!"

        # True = Buyer thắng, False = Seller thắng
        is_refund_buyer = True if "Buyer" in verdict else False
        verdict_text = "BUYER THẮNG" if is_refund_buyer else "SELLER THẮNG"

        print(f"⏳ Đang thực thi: {verdict_text} cho đơn #{order_id}...")

        tx = contract.functions.resolveDispute(int(order_id), is_refund_buyer).build_transaction({
            'from': arbiter.address, 'nonce': w3.eth.get_transaction_count(arbiter.address),
            'gas': 500000, 'gasPrice': w3.eth.gas_price
        })
        signed = w3.eth.account.sign_transaction(tx, arbiter.key)
        w3.eth.wait_for_transaction_receipt(w3.eth.send_raw_transaction(signed.raw_transaction))

        return f"⚖️ ĐÃ TUYÊN ÁN: {verdict_text}!\n(Đã ghi vào Blockchain)"
    except Exception as e: return f"❌ Lỗi thực thi: {e}"

In [ ]:
# --- GIAO DIỆN GRADIO ---
court_theme = gr.themes.Soft(primary_hue="red", secondary_hue="slate").set(
    button_primary_background_fill="*primary_700",
)

with gr.Blocks(title="Arbiter Court", theme=court_theme) as app:
    with gr.Row():
        gr.Markdown("# ⚖️ TÒA ÁN TRỌNG TÀI ONLINE")

    with gr.Tab("1. Hồ Sơ Đang Xử Lý"):
        gr.Markdown("Danh sách các vụ việc đang khiếu nại (Disputed).")
        btn_load = gr.Button("🔄 Tải Danh Sách Khiếu Nại")
        out_cases = gr.Textbox(label="Danh sách hồ sơ", lines=10)
        btn_load.click(get_active_disputes, outputs=out_cases)

    with gr.Tab("2. Phân Xử & Tuyên Án"):
        gr.Markdown("### 🕵️ Bước 1: Thẩm định Bằng chứng")
        with gr.Row(variant="panel"):
            with gr.Column():
                inp_days = gr.Number(label="⏳ Số ngày trôi qua", value=1)
                inp_received = gr.Checkbox(label="📦 Seller xác nhận ĐÃ NHẬN HÀNG?", value=False)
                btn_analyze = gr.Button("🔍 Phân Tích Luật")
            with gr.Column():
                out_suggestion = gr.Textbox(label="Gợi ý Phán Quyết", lines=3)

        btn_analyze.click(analyze_evidence, [inp_days, inp_received], out_suggestion)

        gr.Markdown("### 👨‍⚖️ Bước 2: Ra Phán Quyết")
        with gr.Row(variant="panel"):
            with gr.Column():
                oid = gr.Textbox(label="Mã Vụ Kiện (Order ID)")
                decision = gr.Dropdown(
                    ["Hoàn tiền cho Buyer (Buyer Thắng)", "Trả tiền cho Seller (Seller Thắng)"],
                    label="QUYẾT ĐỊNH CUỐI CÙNG",
                    value="Hoàn tiền cho Buyer (Buyer Thắng)"
                )
                btn_execute = gr.Button("🔨 TUYÊN ÁN (Thực Thi)", variant="primary")
            with gr.Column():
                out_result = gr.Textbox(label="Kết quả thi hành")

        btn_execute.click(execute_judgment, [oid, decision], out_result)

    # --- TAB MỚI: LỊCH SỬ ---
    with gr.Tab("3. Lịch Sử Phán Quyết"):
        gr.Markdown("Danh sách các vụ kiện đã được bạn xử lý xong.")
        btn_hist = gr.Button("📜 Tải Lịch Sử Án Tích")
        out_hist = gr.Textbox(label="Nhật ký phán quyết", lines=15)

        btn_hist.click(view_judgment_history, outputs=out_hist)

app.launch(share=True)

/tmp/ipython-input-4275113772.py:6: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Arbiter Court", theme=court_theme) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c0c078bf1732c67d43.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
